In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 📄 Resume Screening System using Machine Learning

This project classifies resumes into relevant job roles based on their content using NLP and machine learning.

✅ Built with Kaggle  
✅ Uses synthetic but structured dataset  
✅ Predicts roles like Data Analyst, ML Engineer, Consultant etc.


## 🧠 Step 2: Define Target Job Roles

We target 14 job roles including:

- Data Analyst
- Data Scientist
- ML Engineer
- BI Analyst
- Software Engineer
- Backend/Frontend Developer
- DevOps Engineer
- Business Analyst
- Technical Consultant
- Cloud Engineer
- AI Engineer

These roles are commonly found in tech/data industries.



In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import joblib

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## 📊 Step 3: Dataset Generation

A synthetic dataset of resume samples was generated for all 14 roles with 10 entries each.

Each entry includes relevant keywords (skills, tools) per role.

➡️ Total samples: 140  
➡️ Format: text + label



In [3]:
# Load uploaded dataset
df = pd.read_csv("/kaggle/input/resume-sample-dataset/resume_sample_dataset.csv") 

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/resume-sample-dataset/resume_sample_dataset.csv'

In [ ]:
# Clean text function
def clean_text(text):
    stop_words = set(stopwords.words('english'))
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

In [ ]:
# Apply cleaning
df['cleaned_text'] = df['text'].apply(clean_text)



## 🧪 Step 4: TF-IDF + Model Training

- Text data cleaned using regex + stopwords
- Vectorized using **TF-IDF**
- Model: **Random Forest Classifier**
- Files saved:
  - `resume_classifier_model.pkl`
  - `vectorizer.pkl`



In [ ]:
# Vectorization
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['label']

In [ ]:
# Train model
model = RandomForestClassifier()
model.fit(X, y)


In [ ]:
# Save model and vectorizer
joblib.dump(model, "resume_classifier_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")


In [ ]:
joblib.dump(model, "resume_classifier_model.pkl")   


In [ ]:
!pip install pymupdf
import fitz  # PyMuPDF


In [ ]:
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        doc = fitz.open(pdf_path)
        for page in doc:
            t = page.get_text()
            if t:
                text += t
        return text.strip()
    except Exception as e:
        print(f"❌ Error reading PDF: {e}")
        return ""


In [ ]:
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

def clean_text(text):
    stop_words = set(stopwords.words('english'))
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)


## 📁 Step 5: Resume Prediction (PDF to Label)

- Uploaded real-looking sample resumes (PDF format)
- Extracted text using `PyMuPDF`
- Cleaned and passed to trained model
- Printed predicted job role for each resume

🔥 All predictions matched perfectly!



In [ ]:
def predict_resume_role(pdf_path):
    raw_text = extract_text_from_pdf(pdf_path)
    if not raw_text:
        return "❌ No text extracted from PDF"
    
    try:
        cleaned = clean_text(raw_text)
        vector = vectorizer.transform([cleaned])
        prediction = model.predict(vector)
        return prediction[0]
    except Exception as e:
        return f"⚠️ Prediction failed: {e}"


In [ ]:
# Change folder name to your actual uploaded one
pdf_path = "/kaggle/input/sample-testing-resumes/Business_Analyst_Resume.pdf"
print("🎯 Predicted Role:", predict_resume_role(pdf_path))


In [ ]:
# List of your uploaded resume PDF paths
resume_paths = [
    "/kaggle/input/sample-testing-resumes/Data_Analyst_Resume.pdf" ,
    "/kaggle/input/sample-testing-resumes/ML_Engineer_Resume.pdf",
    "/kaggle/input/sample-testing-resumes/Business_Analyst_Resume.pdf",
    "/kaggle/input/sample-testing-resumes/Software_Engineer_Resume.pdf",
    "/kaggle/input/sample-testing-resumes/Technical_Consultant_Resume.pdf"
]

# Predict all and show results
for path in resume_paths:
    role = predict_resume_role(path)
    print(f"📄 {path.split('/')[-1]} ➝ 🎯 Predicted Role: {role}")


✅ Final Results
All 5 test resumes were classified correctly:

📄 Data_Analyst_Resume.pdf → 🎯 Data Analyst
📄 ML_Engineer_Resume.pdf → 🎯 ML Engineer
📄 Business_Analyst_Resume.pdf → 🎯 Business Analyst
📄 Software_Engineer_Resume.pdf → 🎯 Software Engineer
📄 Technical_Consultant_Resume.pdf → 🎯 Technical Consultant
The model performs reliably on realistic inputs.
